In [1]:
import trimesh
import numpy as np
import pytorch_kinematics as pk
import torch
import json
import math
import time
import os
from pathlib import Path
import random
import sys
from pytorch_kinematics.transforms import Transform3d
from pytorch_kinematics.ik import PseudoInverseIK

In [2]:
class D1Model:
    def __init__(self, urdf_path: str, device: str = 'cpu'):
        self.device = device
        self.urdf_path = urdf_path
        # 读取 URDF 构建运动学链
        self.chain = pk.build_chain_from_urdf(open(urdf_path, 'rb').read()).to(dtype=torch.float32, device=device)
        
        # 获取关节数量和名称
        self.joint_names = self.chain.get_joint_parameter_names()
        self.n_dof = len(self.joint_names)

        # --- TCP Offset 设置 (你提供的参数) ---
        self.link6_to_tcp_mat = torch.eye(4, device=device)
        self.link6_to_tcp_mat[0, 3] = 0.0     # x
        self.link6_to_tcp_mat[1, 3] = 0.0     # y
        self.link6_to_tcp_mat[2, 3] = 0.11    # z

        # --- 构建用于IK的SerialChain (只包含6个arm关节，到Link6) ---
        self.serial_chain = pk.SerialChain(self.chain, "Link6", "base_link").to(dtype=torch.float32, device=device)
        self.arm_joint_names = self.serial_chain.get_joint_parameter_names()
        self.arm_n_dof = len(self.arm_joint_names)  # 应该是6
        
        # 从URDF读取的关节限位 (Joint1-Joint6)
        self.joint_limits = torch.tensor([
            [-2.35, 2.35],  # Joint1
            [-1.57, 1.57],  # Joint2
            [-1.57, 1.57],  # Joint3
            [-2.35, 2.35],  # Joint4
            [-1.57, 1.57],  # Joint5
            [-2.35, 2.35],  # Joint6
        ], dtype=torch.float32, device=device)

    def _to_matrix(self, pose):
        """把输入的位姿转成4x4矩阵 (torch, device 对齐)。支持 Transform3d、torch.Tensor 或可转成tensor的array/list。"""
        if isinstance(pose, Transform3d):
            mat = pose.get_matrix()
            return mat.squeeze(0) if mat.dim() == 3 else mat
        if torch.is_tensor(pose):
            mat = pose.to(device=self.device, dtype=torch.float32)
        else:
            mat = torch.tensor(pose, device=self.device, dtype=torch.float32)
        if mat.numel() == 16:
            return mat.reshape(4, 4)
        raise ValueError("target_pose需要是4x4变换矩阵")

    def _to_joint_tensor(self, joints):
        """把输入的关节数组转成 shape=(arm_n_dof,) 的张量。"""
        if torch.is_tensor(joints):
            q = joints.to(device=self.device, dtype=torch.float32).flatten()
        else:
            q = torch.tensor(joints, device=self.device, dtype=torch.float32).flatten()
        if q.numel() != self.arm_n_dof:
            raise ValueError(f"需要{self.arm_n_dof}个关节值, 当前{q.numel()}")
        return q

    def solve_ik(self, target_pose, initial_guess=None, use_tcp_offset=True, pos_tolerance=1e-3, rot_tolerance=1e-2, max_iterations=150, num_retries=8):
        """给定TCP位姿(4x4, base系)求解关节角。默认会减掉TCP offset。"""
        target_mat = self._to_matrix(target_pose)
        if use_tcp_offset:
            tcp_to_link6 = torch.linalg.inv(self.link6_to_tcp_mat)
            link6_target = target_mat @ tcp_to_link6
        else:
            link6_target = target_mat
        target_tf = Transform3d(matrix=link6_target.unsqueeze(0))

        if initial_guess is not None:
            init = self._to_joint_tensor(initial_guess)
            init = torch.clamp(init, self.joint_limits[:, 0], self.joint_limits[:, 1])
            if num_retries < 1:
                num_retries = 1
            if num_retries > 1:
                rand_cfg = torch.rand(num_retries - 1, self.arm_n_dof, device=self.device)
                rand_cfg = rand_cfg * (self.joint_limits[:, 1] - self.joint_limits[:, 0]) + self.joint_limits[:, 0]
                retry_configs = torch.vstack([init, rand_cfg])
            else:
                retry_configs = init.unsqueeze(0)
            solver = PseudoInverseIK(
                self.serial_chain,
                pos_tolerance=pos_tolerance,
                rot_tolerance=rot_tolerance,
                retry_configs=retry_configs,
                joint_limits=self.joint_limits,
                max_iterations=max_iterations,
                lr=0.2,
                config_sampling_method="uniform",
            )
        else:
            solver = PseudoInverseIK(
                self.serial_chain,
                pos_tolerance=pos_tolerance,
                rot_tolerance=rot_tolerance,
                num_retries=num_retries,
                joint_limits=self.joint_limits,
                max_iterations=max_iterations,
                lr=0.2,
                config_sampling_method="uniform",
            )

        sol = solver.solve(target_tf)

        err_score = sol.err_pos[0] + sol.err_rot[0]
        if sol.converged_any[0]:
            mask = sol.converged[0]
            err_score = err_score.masked_fill(~mask, float("inf"))
        best_idx = torch.argmin(err_score)
        best_q = sol.solutions[0, best_idx]
        best_q = torch.clamp(best_q, self.joint_limits[:, 0], self.joint_limits[:, 1])
        return best_q

In [4]:
urdf_path = "assets/urdf/d1_description.urdf"
d1 = D1Model(urdf_path, device='cpu')

In [5]:
# Quick IK self-test: sample random joints -> FK -> IK -> compare
torch.manual_seed(0)
q_gt = d1.joint_limits[:, 0] + torch.rand(d1.arm_n_dof) * (d1.joint_limits[:, 1] - d1.joint_limits[:, 0])
link6_tf = d1.serial_chain.forward_kinematics(q_gt).get_matrix().squeeze(0)
tcp_tf = link6_tf @ d1.link6_to_tcp_mat
q_hat = d1.solve_ik(tcp_tf, use_tcp_offset=True, num_retries=16)
link6_tf_hat = d1.serial_chain.forward_kinematics(q_hat).get_matrix().squeeze(0)
tcp_tf_hat = link6_tf_hat @ d1.link6_to_tcp_mat
pos_err = torch.norm(tcp_tf[:3, 3] - tcp_tf_hat[:3, 3]).item()
cos_theta = ((torch.trace(tcp_tf[:3, :3].T @ tcp_tf_hat[:3, :3]) - 1.0) * 0.5).clamp(-1.0, 1.0)
rot_err = torch.acos(cos_theta).item()
joint_err_max = torch.max(torch.abs(q_hat - q_gt)).item()
print("GT joints(rad):", q_gt.tolist())
print("IK joints(rad):", q_hat.tolist())
print(f"pos_err(m)={pos_err:.6f}, rot_err(rad)={rot_err:.6f}, joint_max_abs_err(rad)={joint_err_max:.6f}")

GT joints(rad): [-0.017594099044799805, 0.8422163724899292, -1.2921808958053589, -1.7294566631317139, -0.6046923995018005, 0.6301698684692383]
IK joints(rad): [-0.017594100907444954, 0.8422151207923889, -1.2921781539916992, -1.7294546365737915, -0.6046921014785767, 0.6301673650741577]
pos_err(m)=0.000000, rot_err(rad)=0.000000, joint_max_abs_err(rad)=0.000003


In [17]:
# Visualize URDF initial configuration: base, link6, TCP, camera and grasp coordinate frames
import numpy as np
import plotly.graph_objects as go

# Get Link6 pose at initial configuration (all joints = 0)
q_init = torch.zeros(d1.arm_n_dof, device=d1.device, dtype=torch.float32)
link6_init_tf = d1.serial_chain.forward_kinematics(q_init).get_matrix().squeeze(0)
T_link6_init = link6_init_tf.cpu().numpy()

# Get TCP pose (Link6 + offset)
link6_to_tcp = d1.link6_to_tcp_mat.cpu().numpy()
T_tcp_init = T_link6_init @ link6_to_tcp

# Camera pose in base frame
T_cam_in_base = np.array([
    [0.04266581138324645, 0.19016429753726094, 0.9808247389218778, 0.3211099820760092],
    [-0.997738937232263, -0.042916397581334996, 0.05172229644240045, 0.054355123827631475],
    [0.051929198623163926, -0.9808138063673455, 0.18790326119989345, 0.47093068126628707],
    [0.0, 0.0, 0.0, 1.0]
])

# Grasp pose in camera frame
T_grasp_in_cam = np.eye(4)
T_grasp_in_cam[:3, 3] = np.array([0.02078194, -0.0152188, 0.20500001])
T_grasp_in_cam[:3, :3] = np.array([
    [-0.01589504, 0.99617106, 0.08596864],
    [-0.18111572, -0.08742573, 0.97956824],
    [0.98333335, 0., 0.18181187]
])

# Transform grasp to base frame
T_grasp_in_base = T_cam_in_base @ T_grasp_in_cam

# Base coordinate frame
T_base = np.eye(4)

def frame_traces(T, name, scale=0.15):
    """Create 3D traces for a coordinate frame (RGB axes)"""
    o = T[:3, 3]
    R = T[:3, :3]
    
    traces = []
    # X-axis (red)
    ax_x = R[:, 0] * scale
    traces.append(go.Scatter3d(
        x=[o[0], o[0] + ax_x[0]], y=[o[1], o[1] + ax_x[1]], z=[o[2], o[2] + ax_x[2]],
        mode='lines+markers', line=dict(color='red', width=8),
        marker=dict(size=4, color='red'), name=f'{name}_X'))
    
    # Y-axis (green)
    ax_y = R[:, 1] * scale
    traces.append(go.Scatter3d(
        x=[o[0], o[0] + ax_y[0]], y=[o[1], o[1] + ax_y[1]], z=[o[2], o[2] + ax_y[2]],
        mode='lines+markers', line=dict(color='green', width=8),
        marker=dict(size=4, color='green'), name=f'{name}_Y'))
    
    # Z-axis (blue)
    ax_z = R[:, 2] * scale
    traces.append(go.Scatter3d(
        x=[o[0], o[0] + ax_z[0]], y=[o[1], o[1] + ax_z[1]], z=[o[2], o[2] + ax_z[2]],
        mode='lines+markers', line=dict(color='blue', width=8),
        marker=dict(size=4, color='blue'), name=f'{name}_Z'))
    
    # Origin point
    traces.append(go.Scatter3d(
        x=[o[0]], y=[o[1]], z=[o[2]], mode='markers',
        marker=dict(size=6, color='black'), name=f'{name}_origin'))
    
    return traces

# Collect all traces
traces = []
traces += frame_traces(T_base, 'base', scale=0.15)
traces += frame_traces(T_link6_init, 'link6', scale=0.12)
traces += frame_traces(T_tcp_init, 'tcp', scale=0.10)
traces += frame_traces(T_cam_in_base, 'camera', scale=0.12)
traces += frame_traces(T_grasp_in_base, 'grasp', scale=0.12)

# Add lines connecting origins
p_base = T_base[:3, 3]
p_link6 = T_link6_init[:3, 3]
p_tcp = T_tcp_init[:3, 3]
p_camera = T_cam_in_base[:3, 3]
p_grasp = T_grasp_in_base[:3, 3]

traces.append(go.Scatter3d(
    x=[p_base[0], p_link6[0]], y=[p_base[1], p_link6[1]], z=[p_base[2], p_link6[2]],
    mode='lines', line=dict(color='gray', width=2, dash='dot'),
    name='base->link6'))

traces.append(go.Scatter3d(
    x=[p_link6[0], p_tcp[0]], y=[p_link6[1], p_tcp[1]], z=[p_link6[2], p_tcp[2]],
    mode='lines', line=dict(color='purple', width=2, dash='dash'),
    name='link6->tcp'))

traces.append(go.Scatter3d(
    x=[p_base[0], p_camera[0]], y=[p_base[1], p_camera[1]], z=[p_base[2], p_camera[2]],
    mode='lines', line=dict(color='cyan', width=2, dash='dot'),
    name='base->camera'))

traces.append(go.Scatter3d(
    x=[p_camera[0], p_grasp[0]], y=[p_camera[1], p_grasp[1]], z=[p_camera[2], p_grasp[2]],
    mode='lines', line=dict(color='orange', width=2, dash='dash'),
    name='camera->grasp'))

# Create figure
fig = go.Figure(data=traces)
fig.update_layout(
    scene=dict(aspectmode='data', xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
    title='Robot Configuration: Base, Link6, TCP, Camera and Grasp Frames'
)
fig.show()

In [ ]:
# Transform grasp: rotate 90° about own Y-axis, then 180° about own X-axis
import numpy as np
import plotly.graph_objects as go

# Original grasp in base frame (from previous cell)
T_grasp_orig = T_grasp_in_base.copy()

# Create rotation matrices for grasp frame transformations
# Rotation about Y-axis by 90 degrees
theta_y = - np.pi / 2  # 90 degrees
Ry_90 = np.array([
    [np.cos(theta_y), 0, np.sin(theta_y), 0],
    [0, 1, 0, 0],
    [-np.sin(theta_y), 0, np.cos(theta_y), 0],
    [0, 0, 0, 1]
])

# Rotation about X-axis by 180 degrees
theta_x = np.pi  # 180 degrees
Rx_180 = np.array([
    [1, 0, 0, 0],
    [0, np.cos(theta_x), -np.sin(theta_x), 0],
    [0, np.sin(theta_x), np.cos(theta_x), 0],
    [0, 0, 0, 1]
])

# Apply rotations: first Y-90, then X+180 (applied in grasp's own frame)
T_grasp_rotated = T_grasp_orig @ Ry_90 @ Rx_180

def frame_traces(T, name, scale=0.12):
    """Create 3D traces for a coordinate frame (RGB axes)"""
    o = T[:3, 3]
    R = T[:3, :3]
    
    traces = []
    # X-axis (red)
    ax_x = R[:, 0] * scale
    traces.append(go.Scatter3d(
        x=[o[0], o[0] + ax_x[0]], y=[o[1], o[1] + ax_x[1]], z=[o[2], o[2] + ax_x[2]],
        mode='lines+markers', line=dict(color='red', width=8),
        marker=dict(size=4, color='red'), name=f'{name}_X'))
    
    # Y-axis (green)
    ax_y = R[:, 1] * scale
    traces.append(go.Scatter3d(
        x=[o[0], o[0] + ax_y[0]], y=[o[1], o[1] + ax_y[1]], z=[o[2], o[2] + ax_y[2]],
        mode='lines+markers', line=dict(color='green', width=8),
        marker=dict(size=4, color='green'), name=f'{name}_Y'))
    
    # Z-axis (blue)
    ax_z = R[:, 2] * scale
    traces.append(go.Scatter3d(
        x=[o[0], o[0] + ax_z[0]], y=[o[1], o[1] + ax_z[1]], z=[o[2], o[2] + ax_z[2]],
        mode='lines+markers', line=dict(color='blue', width=8),
        marker=dict(size=4, color='blue'), name=f'{name}_Z'))
    
    # Origin point
    traces.append(go.Scatter3d(
        x=[o[0]], y=[o[1]], z=[o[2]], mode='markers',
        marker=dict(size=6, color='black'), name=f'{name}_origin'))
    
    return traces

# Collect traces for both original and rotated grasp
traces = []
traces += frame_traces(T_grasp_orig, 'grasp_orig', scale=0.12)
traces += frame_traces(T_grasp_rotated, 'grasp_rotated', scale=0.12)

# Connect the two origins
p_orig = T_grasp_orig[:3, 3]
p_rotated = T_grasp_rotated[:3, 3]
traces.append(go.Scatter3d(
    x=[p_orig[0], p_rotated[0]], y=[p_orig[1], p_rotated[1]], z=[p_orig[2], p_rotated[2]],
    mode='lines', line=dict(color='gray', width=2, dash='dot'),
    name='orig->rotated'))

# Create figure
fig = go.Figure(data=traces)
fig.update_layout(
    scene=dict(aspectmode='data', xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
    title='Grasp Transformation: Original vs Rotated (Y+90°, X+180°)'
)
fig.show()

In [20]:
# Visualize: Robot base, camera, TCP, and rotated grasp
import numpy as np
import plotly.graph_objects as go

# Base coordinate frame
T_base = np.eye(4)

# TCP pose (Link6 + offset at initial config)
T_tcp = T_tcp_init.copy()

# Camera pose
T_camera = T_cam_in_base.copy()

# Rotated grasp (from previous cell)
T_grasp = T_grasp_rotated.copy()

def frame_traces(T, name, scale=0.12):
    """Create 3D traces for a coordinate frame (RGB axes)"""
    o = T[:3, 3]
    R = T[:3, :3]
    
    traces = []
    # X-axis (red)
    ax_x = R[:, 0] * scale
    traces.append(go.Scatter3d(
        x=[o[0], o[0] + ax_x[0]], y=[o[1], o[1] + ax_x[1]], z=[o[2], o[2] + ax_x[2]],
        mode='lines+markers', line=dict(color='red', width=8),
        marker=dict(size=4, color='red'), name=f'{name}_X'))
    
    # Y-axis (green)
    ax_y = R[:, 1] * scale
    traces.append(go.Scatter3d(
        x=[o[0], o[0] + ax_y[0]], y=[o[1], o[1] + ax_y[1]], z=[o[2], o[2] + ax_y[2]],
        mode='lines+markers', line=dict(color='green', width=8),
        marker=dict(size=4, color='green'), name=f'{name}_Y'))
    
    # Z-axis (blue)
    ax_z = R[:, 2] * scale
    traces.append(go.Scatter3d(
        x=[o[0], o[0] + ax_z[0]], y=[o[1], o[1] + ax_z[1]], z=[o[2], o[2] + ax_z[2]],
        mode='lines+markers', line=dict(color='blue', width=8),
        marker=dict(size=4, color='blue'), name=f'{name}_Z'))
    
    # Origin point
    traces.append(go.Scatter3d(
        x=[o[0]], y=[o[1]], z=[o[2]], mode='markers',
        marker=dict(size=6, color='black'), name=f'{name}_origin'))
    
    return traces

# Collect all traces
traces = []
traces += frame_traces(T_base, 'base', scale=0.15)
traces += frame_traces(T_camera, 'camera', scale=0.12)
traces += frame_traces(T_tcp, 'tcp', scale=0.10)
traces += frame_traces(T_grasp, 'grasp_rotated', scale=0.12)

# Add lines connecting origins
p_base = T_base[:3, 3]
p_camera = T_camera[:3, 3]
p_tcp = T_tcp[:3, 3]
p_grasp = T_grasp[:3, 3]

traces.append(go.Scatter3d(
    x=[p_base[0], p_camera[0]], y=[p_base[1], p_camera[1]], z=[p_base[2], p_camera[2]],
    mode='lines', line=dict(color='cyan', width=2, dash='dot'),
    name='base->camera'))

traces.append(go.Scatter3d(
    x=[p_base[0], p_tcp[0]], y=[p_base[1], p_tcp[1]], z=[p_base[2], p_tcp[2]],
    mode='lines', line=dict(color='purple', width=2, dash='dash'),
    name='base->tcp'))

traces.append(go.Scatter3d(
    x=[p_camera[0], p_grasp[0]], y=[p_camera[1], p_grasp[1]], z=[p_camera[2], p_grasp[2]],
    mode='lines', line=dict(color='orange', width=2, dash='dash'),
    name='camera->grasp_rotated'))

# Create figure
fig = go.Figure(data=traces)
fig.update_layout(
    scene=dict(aspectmode='data', xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
    title='Robot Configuration: Base, Camera, TCP and Rotated Grasp'
)
fig.show()

In [22]:
# Solve IK for the rotated grasp pose
import torch
import numpy as np

# Convert T_grasp_rotated to torch tensor
T_grasp_rotated_torch = torch.from_numpy(T_grasp_rotated).float()

# Solve IK with multiple retries
print("Solving IK for rotated grasp pose...")
q_grasp_ik = d1.solve_ik(
    T_grasp_rotated_torch,
    use_tcp_offset=True,
    pos_tolerance=1e-4,
    rot_tolerance=1e-3,
    max_iterations=200,
    num_retries=32
)

print(f"\nSolved joint angles (radians): {q_grasp_ik.tolist()}")
print(f"Solved joint angles (degrees):  {(q_grasp_ik * 180 / np.pi).tolist()}")

# Verify the solution with forward kinematics
link6_fk = d1.serial_chain.forward_kinematics(q_grasp_ik).get_matrix().squeeze(0)
tcp_fk = (link6_fk @ d1.link6_to_tcp_mat).cpu().numpy()

# Calculate position and rotation errors
pos_diff = T_grasp_rotated[:3, 3] - tcp_fk[:3, 3]
pos_error = np.linalg.norm(pos_diff)

# Calculate rotation error using trace
cos_trace = (np.trace(T_grasp_rotated[:3, :3].T @ tcp_fk[:3, :3]) - 1) / 2
cos_trace = np.clip(cos_trace, -1.0, 1.0)
rot_error = np.arccos(cos_trace)

print(f"\nVerification Results:")
print(f"Position error (m): {pos_error:.6f}")
print(f"Rotation error (rad): {rot_error:.6f}")
print(f"Rotation error (deg): {rot_error * 180 / np.pi:.4f}")

# Print target and achieved TCP poses
print(f"\nTarget TCP position: {T_grasp_rotated[:3, 3]}")
print(f"Achieved TCP position: {tcp_fk[:3, 3]}")

# Check if solution is within joint limits
within_limits = torch.all(
    (q_grasp_ik >= d1.joint_limits[:, 0]) & (q_grasp_ik <= d1.joint_limits[:, 1])
)
print(f"\nWithin joint limits: {within_limits.item()}")

Solving IK for rotated grasp pose...



Solved joint angles (radians): [-2.3499999046325684, -1.5700000524520874, 1.5700000524520874, -2.3499999046325684, -1.5700000524520874, 2.3499999046325684]
Solved joint angles (degrees):  [-134.6450653076172, -89.95437622070312, 89.95437622070312, -134.6450653076172, -89.95437622070312, 134.6450653076172]

Verification Results:
Position error (m): 0.630548
Rotation error (rad): 1.510735
Rotation error (deg): 86.5587

Target TCP position: [0.52017167 0.04487638 0.52545685]
Achieved TCP position: [0.14787199 0.04739607 0.01655816]

Within joint limits: True
